# LSTM Colab Notebook

In [ ]:
#Import files from Google Drive (add your own paths)
import os
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)
os.chdir('/content/gdrive/MyDrive/ColabNotebooks/LSTM')
snapshot_dir = '/content/gdrive/MyDrive/ColabNotebooks/LSTM/lstm_snapshots'
os.makedirs(snapshot_dir, exist_ok=True)
print(f"Snapshots will be saved to: {snapshot_dir}")

In [ ]:
!unzip /content/gdrive/MyDrive/ColabNotebooks/guitar_dataset.zip -d /content/guitar_dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
import os
import time

from lstm_training import LSTMTrainer
from lstm_logging import TensorboardLogger
from lstm_model import LSTMModel
from lstm_loss import LSTMCombinedLoss
from lstm_paired_audio_data import PairedLSTMDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
SAMPLING_RATE = 16000
DATASET_FILE = '/content/guitar_dataset/lstm_paired_dataset.npz'
CLEAN_DIR = '/content/guitar_dataset/clean_guitar'
PROCESSED_DIR = '/content/guitar_dataset/overdrive_guitar'

SEQUENCE_LENGTH = 8192
HIDDEN_SIZE = 128
NUM_LAYERS = 2
BIDIRECTIONAL = False
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 300
SNAPSHOT_INTERVAL = 400

In [ ]:
dataset = PairedLSTMDataset(
    dataset_file=DATASET_FILE,
    item_length=SEQUENCE_LENGTH,
    clean_dir=CLEAN_DIR,
    processed_dir=PROCESSED_DIR,
    target_length=SEQUENCE_LENGTH,
    sampling_rate=SAMPLING_RATE,
    device=device
)

In [ ]:
#Initialize Model
model = LSTMModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    output_size=1,
    bidirectional=BIDIRECTIONAL,
    dropout=0.2,
    device=device).to(device)

In [ ]:
# Restore Model State and Resume Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = '/content/gdrive/MyDrive/ColabNotebooks/LSTM/lstm_snapshots/snapshot_2026-04-01_10-42-12.pth' 

ckpt = torch.load(model_path, map_location=device)

# Load model, optimizer and scheduler
model.load_state_dict(ckpt['model_state_dict'])
trainer.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
if 'scheduler_state_dict' in ckpt:
    trainer.scheduler.load_state_dict(ckpt['scheduler_state_dict'])

start_epoch = ckpt.get('epoch', 0) + 1

model.train()

print(f"Training Resumed from Epoch {start_epoch}")
print(f"Model restored from: {model_path}")

In [ ]:
# Initialize Logger and Trainer
logger = TensorboardLogger(log_interval=50, validation_interval=100)
criterion = LSTMCombinedLoss()

trainer = LSTMTrainer(
    model=model,
    dataset=dataset,
    lr=LEARNING_RATE,
    gradient_clipping=1.0,
    logger=logger,
    snapshot_path=snapshot_dir,
    snapshot_interval=SNAPSHOT_INTERVAL,
    device=device
)

logger.trainer = trainer

trainer.train(batch_size=BATCH_SIZE, epochs=EPOCHS)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = '/content/gdrive/MyDrive/ColabNotebooks/LSTM/lstm_snapshots/snapshot_2026-04-04_14-39-59.pth'


ckpt = torch.load(model_path, map_location=device)

model.load_state_dict(ckpt['model_state_dict'])

model = model.to(device)
model.eval()

print(f"Model restored from: {model_path}")
print("Status: Set to eval mode.")

In [ ]:
#.wav generation
import torch
import torchaudio
from IPython.display import Audio

input_wav_path = '/content/gdrive/MyDrive/ColabNotebooks/2-0.wav'
output_wav_path = '/content/gdrive/MyDrive/ColabNotebooks/LSTM/lstm_generated_audio/lstm_processed_2-0.wav'
print(f"Processing: {input_wav_path}")


waveform, sample_rate = torchaudio.load(input_wav_path)
waveform = waveform.to(device).float()


if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)


with torch.no_grad():
    
    input_tensor = waveform.transpose(0, 1).unsqueeze(0).contiguous()

    with torch.backends.cudnn.flags(enabled=False):
        output_tensor = model(input_tensor)

    output_waveform = output_tensor.squeeze(0).transpose(0, 1)


output_waveform = output_waveform.cpu()
torchaudio.save(output_wav_path, output_waveform, sample_rate)


print(f"Success! Saved to: {output_wav_path}")

Audio(output_wav_path)